# 2.0 Refactorización de variables (versión final)

Paso intermedio entre la exploración (`1.0-p2-exploracion-mibici`) y `features.py`. Versión alineada con la corrección de aislamiento: X solo lleva calendario y la partición se verifica contra el manifiesto confirmado. La versión anterior, con rezagos, sigue en `notebooks/2.0-dm-features-refactorizacion.ipynb`.

In [1]:
import hashlib
import json

import numpy as np
import pandas as pd

from mibici_equipo39 import config as C

departures = pd.read_parquet(C.INTERIM / "departures", columns=["station_id", "timestamp"])
arrivals = pd.read_parquet(C.INTERIM / "arrivals", columns=["station_id", "timestamp"])
stations = pd.read_parquet(C.PROCESSED / "stations.parquet")
len(departures), len(arrivals), len(stations)

(4497119, 4497119, 484)

## Eventos y malla estación-hora

In [2]:
def make_events(clean_departures, clean_arrivals):
    departures = clean_departures[["station_id", "timestamp"]].assign(event=-1)
    arrivals = clean_arrivals[["station_id", "timestamp"]].assign(event=1)
    events = pd.concat([departures, arrivals], ignore_index=True)
    events["station_id"] = events["station_id"].astype("string")
    return events.sort_values(["timestamp", "station_id"]).reset_index(drop=True)


def aggregate_hourly(events, stations, start=C.SOURCE_START, end=C.SOURCE_END_EXCLUSIVE):
    df = events.loc[events["station_id"].isin(stations["station_id"])].copy()
    df["hour"] = df["timestamp"].dt.floor("h")
    df = df.loc[df["hour"].ge(start) & df["hour"].lt(end)]
    counts = (df.groupby(["station_id", "hour", "event"], observed=True).size()
              .unstack("event", fill_value=0).reindex(columns=[-1, 1], fill_value=0)
              .rename(columns={-1: "departures", 1: "arrivals"}))
    index = pd.MultiIndex.from_product(
        [sorted(df["station_id"].unique()), pd.date_range(start, end, freq="h", inclusive="left")],
        names=["station_id", "hour"])
    hourly = counts.reindex(index, fill_value=0).reset_index()
    hourly.columns.name = None
    hourly["station_id"] = hourly["station_id"].astype("string")
    return hourly

events = make_events(departures, arrivals)
hourly = aggregate_hourly(events, stations)
print(hourly["station_id"].nunique(), "estaciones x", hourly["hour"].nunique(), "horas =", len(hourly))
print("horas sin salidas:", round(hourly["departures"].eq(0).mean(), 3))

469 estaciones x 8760 horas = 4108440
horas sin salidas: 0.621


> **Por qué la malla completa.** Más de la mitad de las horas no tiene salidas. Si solo se agregan las horas con viajes, esos ceros desaparecen y el modelo sobreestima la demanda. El rango queda fijo en `config.py` para que las llegadas posteriores al 30 de septiembre no agreguen horas.

## Calendario

In [3]:
def add_calendar_features(df):
    df = df.copy()
    df["hour_of_day"] = df["hour"].dt.hour
    df["day_of_week"] = df["hour"].dt.dayofweek
    df["month"] = df["hour"].dt.month
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
    df["hour_sin"] = np.sin(2 * np.pi * df["hour_of_day"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour_of_day"] / 24)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    return df

hourly = add_calendar_features(hourly)
hourly.groupby("hour_of_day")["departures"].mean().round(2).to_frame("salidas_promedio").T

hour_of_day,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
salidas_promedio,0.17,0.0,0.0,0.0,0.0,0.23,0.77,1.48,1.97,1.45,...,1.73,1.77,1.68,1.94,2.34,1.8,1.37,1.08,0.65,0.33


> **Por qué calendario.** Las salidas cambian mucho por hora y por día; esas variables se conocen antes de la hora que se predice. Las versiones seno y coseno le dicen al modelo que las 23 h y las 0 h están juntas.

## Partición por días completos

In [4]:
def random_grouped_split(df, seed=C.SEED, data_version="1"):
    df = df.copy()
    df["date"] = df["hour"].dt.normalize()
    rng = np.random.default_rng(seed)
    assignments = {}
    for _, days in df.groupby(df["date"].dt.to_period("M"))["date"]:
        days = np.sort(days.unique())
        rng.shuffle(days)
        cut_train = int(np.floor(C.TRAIN_SIZE * len(days) + 0.5))
        cut_val = int(np.floor((C.TRAIN_SIZE + C.VAL_SIZE) * len(days) + 0.5))
        for i, day in enumerate(days):
            assignments[pd.Timestamp(day)] = "train" if i < cut_train else "validation" if i < cut_val else "test"
    df["split"] = df["date"].map(assignments)
    days = {s: sorted(d.strftime("%Y-%m-%d") for d, v in assignments.items() if v == s) for s in ["train", "validation", "test"]}
    content = {"seed": seed, "strategy": C.SPLIT_STRATEGY, "data_version": data_version, "days": days}
    manifest = {"split_id": hashlib.sha256(json.dumps(content, sort_keys=True).encode()).hexdigest(), **content}
    return df.drop(columns="date"), manifest

data_version = json.loads((C.REFERENCES / "data_manifest.json").read_text())["data_version"]
hourly = hourly.loc[hourly["hour"].ge(C.FEATURES_START)].reset_index(drop=True)
split_df, manifest = random_grouped_split(hourly, data_version=data_version)
confirmed = json.loads((C.REFERENCES / "split_manifest.json").read_text())
print("coincide con split_manifest.json confirmado:", manifest == confirmed)
pd.Series({k: len(v) for k, v in manifest["days"].items()}).rename("días").to_frame().assign(proporción=lambda d: (d["días"] / d["días"].sum()).round(3))

coincide con split_manifest.json confirmado: True


,días,proporción
train,252,0.706
validation,52,0.146
test,53,0.148


> **Por qué días completos al azar y verificados.** Las horas seguidas de un día se parecen mucho; repartir filas sueltas dejaría la 9:00 en prueba con la 8:00 y la 10:00 en train. El sorteo es por mes para que los doce meses aparezcan en los tres conjuntos. El `split_id` es el hash del mapa de días y la semilla, y la construcción compara contra el manifiesto confirmado en Git: si algo cambia, se detiene en lugar de reescribirlo.

## Por qué no hay rezagos en X

In [5]:
ordered = hourly.sort_values(["station_id", "hour"]).reset_index(drop=True)
by_station = ordered.groupby("station_id", sort=False, observed=True)["departures"]
print("correlación de las salidas con su valor de hace 24 h:", round(ordered["departures"].corr(by_station.shift(24)), 3))

day_split = split_df.drop_duplicates("hour").set_index(split_df.drop_duplicates("hour")["hour"].dt.normalize())["split"]
day_split = day_split[~day_split.index.duplicated()]
train_days = day_split[day_split.eq("train")].index
previous = (train_days - pd.Timedelta("1D")).intersection(day_split.index)
print("días de train cuyo día anterior está en validación o prueba:",
      f"{day_split.loc[previous].ne('train').mean():.0%}")

correlación de las salidas con su valor de hace 24 h: 0.644
días de train cuyo día anterior está en validación o prueba: 29%


> **Los rezagos ayudarían, pero filtrarían datos reservados.** Las salidas de hace 24 horas explican bien la hora siguiente. Pero con días al azar, casi un tercio de los días de train tiene el día anterior en validación o prueba: `dep_lag_24` de esas filas sería un objetivo reservado metido en el entrenamiento. Por eso X lleva solo calendario. `add_lagged_features` sigue en `features.py` para una futura partición temporal, donde sí se pueden usar.

## Comparación con el módulo

In [6]:
from mibici_equipo39 import features as F

module = F.add_calendar_features(F.aggregate_hourly(F.make_events(departures, arrivals), stations))
module = module.loc[module["hour"].ge(C.FEATURES_START)].reset_index(drop=True)
module, module_manifest = F.random_grouped_split(module, manifest_path=F.SPLIT_PATH, data_version=data_version)
cols = ["station_id", "hour", "departures", "arrivals", "split", *F.FEATURE_COLUMNS]
pd.testing.assert_frame_equal(split_df[cols].reset_index(drop=True), module[cols].reset_index(drop=True), check_dtype=False)
print("notebook y módulo coinciden en", len(module), "filas; X =", F.FEATURE_COLUMNS)

notebook y módulo coinciden en 4018392 filas; X = ['hour_of_day', 'day_of_week', 'month', 'is_weekend', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos']
